### Flipkart Web Data Scraping

#### **Problem statement:** 
- Analyze laptop prices, specs, ratings, and reviews to understand market trends and customer preferences.



# Key Points

## 1. Project Context
- **Goal:** Build a recruiter‑ready dataset of laptops scraped from Flipkart, cleaned to ≥400 rows after deduplication.
- **Why:** Demonstrates ability to handle real‑world messy data, document cleaning choices, and extract business insights.
- **Deliverables:** Clean dataset + EDA (univariate, bivariate, multivariate) with clear visualizations.

---

## 2. Data Pipeline Steps
### Step 1: Import & Copy
- Always work on a copy (`df = df_laptops.copy()`).
- Prevents accidental overwriting of raw data.

### Step 2: Column Renaming & Conversion
- Renamed raw scraped columns → target schema (`Product_Title`, `Price`, `Rating`, etc.).
- Converted:
  - `Price`: ₹ string → float.
  - `Rating`: string → float.
  - `Ratings_Count`, `Reviews_Count`: string with commas → int.

### Step 3: Brand Extraction
- Extracted `Brand_Name` from first word of `Product_Title`.

### Step 4: Regex Parsing from Specifications
- Extracted **RAM_GB, Storage_GB, CPU_Brand, Processor_Name, Display_Size_Inch, Touchscreen, Operating_System, RAM_Type**.
- Robust regex handles multiple formats (e.g., “1 TB + 256 GB”).

### Step 5: Deduplication
- Dropped exact duplicates across all columns.
- Preserved seller variation (same title/specs but different price/ratings).

### Step 6: Missing Values
- **Numerical:** Median imputation; Chromebooks storage → 0 if missing.
- **Categorical:** Filled with `"Unknown"` for transparency.

### Step 7: Feature Engineering
- **Price_Tier:** Budget (<45k), Mid‑range (45–80k), Premium (≥80k).
- **RAM_Category:** Low (<8 GB), Mid (8–12 GB), High (16+ GB).
- **Storage_Category:** Low (<512 GB), Mid (512 GB), High (≥1 TB).

### Step 8: Column Ordering
- Target variables first (Brand, Price, RAM, Storage, CPU, OS, Ratings).
- Reference variables (`Product_Title`, `Specifications`) at the end.

---

In [2]:
# Step 1: Import required libraries
import requests
import pandas as pd
import re
from bs4 import BeautifulSoup

In [3]:
# Step 2: Define headers to mimic a real browser
headers = {
    'user-agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/143.0.0.0 Safari/537.36',
    'accept-encoding': 'gzip, deflate, br, zstd'
}

In [4]:
# Create an empty list to store laptop data
data_laptops = []

In [5]:
import time

for page in range(1, 6):
    print(f"Scraping page {page}...")
    url = f"https://www.flipkart.com/search?q=laptops&page={page}"
    response = requests.get(url, headers=headers, timeout=10)
    soup = BeautifulSoup(response.text, "html.parser")
    # your scraping logic...
    time.sleep(2)   # wait 2 seconds before next request

Scraping page 1...
Scraping page 2...
Scraping page 3...
Scraping page 4...
Scraping page 5...


In [6]:
# Step 3 : Scrape more pages for larger dataset
data_laptops = []

for page in range(1, 45):  # scrape first 20 pages (~480 products)
    print(f"Scraping page {page}...")
    url = f"https://www.flipkart.com/search?q=laptops&page={page}"
    response = requests.get(url, headers=headers)
    soup = BeautifulSoup(response.text, "html.parser")

    titles = soup.find_all("div", class_="RG5Slk")

    for t in titles:
        title = t.get_text(strip=True)

        # Price
        price_tag = t.find_next("div", class_="hZ3P6w DeU9vF")
        price = price_tag.get_text(strip=True) if price_tag else None

        # Specs
        specs_tag = t.find_next("div", class_="CMXw7N")
        specs = None
        if specs_tag:
            specs_list = specs_tag.find_all("li", class_="DTBslk")
            specs = " | ".join([li.get_text(strip=True) for li in specs_list])

        # Reviews
        reviews_tag = t.find_next("div", class_="a7saXW")
        rating, ratings_count, reviews_count = None, None, None
        if reviews_tag:
            rating_div = reviews_tag.find("div", class_="MKiFS6")
            rating = rating_div.get_text(strip=True) if rating_div else None

            counts_span = reviews_tag.find("span", class_="PvbNMB")
            if counts_span:
                counts_text = counts_span.get_text(" ", strip=True)
                parts = counts_text.split("&")
                if len(parts) >= 2:
                    ratings_count = parts[0].replace("Ratings", "").strip()
                    reviews_count = parts[1].replace("Reviews", "").strip()

        data_laptops.append({
            "title": title,
            "price": price,
            "rating": rating,
            "ratings_count": ratings_count,
            "reviews_count": reviews_count,
            "specs": specs
        })

# Convert to DataFrame with new name
df_laptops = pd.DataFrame(data_laptops)
print(df_laptops.head())
print("Total rows scraped:", len(df_laptops))

Scraping page 1...
Scraping page 2...
Scraping page 3...
Scraping page 4...
Scraping page 5...
Scraping page 6...
Scraping page 7...
Scraping page 8...
Scraping page 9...
Scraping page 10...
Scraping page 11...
Scraping page 12...
Scraping page 13...
Scraping page 14...
Scraping page 15...
Scraping page 16...
Scraping page 17...
Scraping page 18...
Scraping page 19...
Scraping page 20...
Scraping page 21...
Scraping page 22...
Scraping page 23...
Scraping page 24...
Scraping page 25...
Scraping page 26...
Scraping page 27...
Scraping page 28...
Scraping page 29...
Scraping page 30...
Scraping page 31...
Scraping page 32...
Scraping page 33...
Scraping page 34...
Scraping page 35...
Scraping page 36...
Scraping page 37...
Scraping page 38...
Scraping page 39...
Scraping page 40...
Scraping page 41...
Scraping page 42...
Scraping page 43...
Scraping page 44...
                                               title      price rating  \
0  ASUS Vivobook 16X (2025) for Creator with Offi...   

In [8]:
df_laptops.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 984 entries, 0 to 983
Data columns (total 6 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   title          984 non-null    object
 1   price          984 non-null    object
 2   rating         981 non-null    object
 3   ratings_count  981 non-null    object
 4   reviews_count  981 non-null    object
 5   specs          984 non-null    object
dtypes: object(6)
memory usage: 46.3+ KB


In [13]:
# Save df_laptops to a CSV file
df_laptops.to_csv("raw_scraped_fk_laptops_data.csv", index=False, encoding="utf-8")

In [15]:
import pandas as pd

# Work on a copy
df = df_laptops.copy()

# Rename columns
df = df.rename(columns={
    "title": "Product_Title",
    "price": "Price",
    "rating": "Rating",
    "ratings_count": "Ratings_Count",
    "reviews_count": "Reviews_Count",
    "specs": "Specifications"
})

# Convert Price from ₹ string to float
df["Price"] = (df["Price"]
               .str.replace("₹", "", regex=False)
               .str.replace(",", "", regex=False)
               .astype(float))
# Ratings_Count: remove commas, coerce to numeric, then fill missing with 0
df["Ratings_Count"] = (df["Ratings_Count"]
                       .str.replace(",", "", regex=False)
                       .replace("", "0")  # handle empty strings
                       .pipe(pd.to_numeric, errors="coerce")
                       .fillna(0)
                       .astype(int))

# Reviews_Count
df["Reviews_Count"] = (df["Reviews_Count"]
                       .str.replace(",", "", regex=False)
                       .replace("", "0")
                       .pipe(pd.to_numeric, errors="coerce")
                       .fillna(0)
                       .astype(int))


# Extract Brand_Name from Product_Title (first word)
df["Brand_Name"] = df["Product_Title"].str.split().str[0]

# Add empty placeholders for columns to be filled later
for col in ["RAM_GB","Storage_GB","CPU_Brand","Display_Size_Inch",
            "Touchscreen","Operating_System","Processor_Name","RAM_Type",
            "Price_Tier","RAM_Category","Storage_Category"]:
    df[col] = None



In [16]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 984 entries, 0 to 983
Data columns (total 18 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Product_Title      984 non-null    object 
 1   Price              984 non-null    float64
 2   Rating             981 non-null    object 
 3   Ratings_Count      984 non-null    int64  
 4   Reviews_Count      984 non-null    int64  
 5   Specifications     984 non-null    object 
 6   Brand_Name         984 non-null    object 
 7   RAM_GB             0 non-null      object 
 8   Storage_GB         0 non-null      object 
 9   CPU_Brand          0 non-null      object 
 10  Display_Size_Inch  0 non-null      object 
 11  Touchscreen        0 non-null      object 
 12  Operating_System   0 non-null      object 
 13  Processor_Name     0 non-null      object 
 14  RAM_Type           0 non-null      object 
 15  Price_Tier         0 non-null      object 
 16  RAM_Category       0 non-n

In [17]:
df.head()

,Product_Title,Price,Rating,Ratings_Count,Reviews_Count,Specifications,Brand_Name,RAM_GB,Storage_GB,CPU_Brand,Display_Size_Inch,Touchscreen,Operating_System,Processor_Name,RAM_Type,Price_Tier,RAM_Category,Storage_Category
0,ASUS Vivobook 16X (2025) for Creator with Offi...,62990.0,4.3,1318,74,Intel Core i5 Processor (13th Gen) | 16 GB DDR...,ASUS,None,None,None,None,None,None,None,None,None,None,None
1,ASUS TUF Gaming A15 (2025) AMD Ryzen 7 Hexa Co...,66990.0,4.4,3096,169,AMD Ryzen 7 Hexa Core Processor | 16 GB DDR5 R...,ASUS,None,None,None,None,None,None,None,None,None,None,None
2,ASUS Vivobook Go 15 AMD Ryzen 3 Quad Core 7320...,32190.0,4.3,1747,116,AMD Ryzen 3 Quad Core Processor | 8 GB LPDDR5 ...,ASUS,None,None,None,None,None,None,None,None,None,None,None
3,Lenovo IdeaPad Pro 5 Intel Core Ultra 9 285H -...,108990.0,4.2,6,0,Intel Core Ultra 9 Processor | 32 GB LPDDR5X R...,Lenovo,None,None,None,None,None,None,None,None,None,None,None
4,Acer Aspire 3 Intel Celeron Dual Core - (8 GB/...,23590.0,3.8,8033,700,Intel Celeron Dual Core Processor | 8 GB DDR4 ...,Acer,None,None,None,None,None,None,None,None,None,None,None


In [35]:
import re
import numpy as np

# --- RAM in GB ---
def parse_ram(text):
    if not isinstance(text, str): return np.nan
    m = re.findall(r'(\d+(?:\.\d+)?)\s*GB', text, flags=re.I)
    return max([float(x) for x in m]) if m else np.nan

# --- Storage in GB ---
def parse_storage(text):
    if not isinstance(text, str): return np.nan
    total = 0
    tb_matches = re.findall(r'(\d+(?:\.\d+)?)\s*TB', text, flags=re.I)
    gb_matches = re.findall(r'(\d+(?:\.\d+)?)\s*GB', text, flags=re.I)
    for tb in tb_matches: total += float(tb) * 1024
    for gb in gb_matches: total += float(gb)
    return total if total > 0 else np.nan

# --- CPU Brand ---
def parse_cpu_brand(text):
    if not isinstance(text, str): return "Unknown"
    t = text.lower()
    if "intel" in t: return "Intel"
    if "amd" in t: return "AMD"
    if "apple" in t: return "Apple"
    if "snapdragon" in t or "qualcomm" in t: return "Qualcomm"
    return "Unknown"

# --- Processor Name ---
def parse_processor(text):
    if not isinstance(text, str): return "Unknown"
    m = re.search(r'(Intel.*?Processor|AMD.*?Processor|Apple.*?M\d+|Snapdragon.*?Processor)', text, flags=re.I)
    return m.group(0) if m else "Unknown"

# --- Display Size ---
def parse_display(text):
    if not isinstance(text, str): return np.nan
    m = re.findall(r'(\d{1,2}(?:\.\d)?)\s*(?:inch|in|\"|”)', text, flags=re.I)
    return float(m[0]) if m else np.nan

# --- Touchscreen ---
def parse_touch(text):
    if not isinstance(text, str): return "Unknown"
    t = text.lower()
    if "touch" in t: return "Yes"
    return "No"

# --- Operating System ---
def parse_os(text):
    if not isinstance(text, str): return "Unknown"
    t = text.lower()
    if "windows" in t: return "Windows"
    if "chrome" in t: return "Chrome OS"
    if "mac" in t: return "macOS"
    if "linux" in t or "ubuntu" in t: return "Linux"
    return "Unknown"

# --- RAM Type ---
def parse_ram_type(text):
    if not isinstance(text, str): return "Unknown"
    t = text.lower()
    if "ddr5" in t or "lpddr5" in t: return "DDR5/LPDDR5"
    if "ddr4" in t or "lpddr4" in t: return "DDR4/LPDDR4"
    if "ddr3" in t or "lpddr3" in t: return "DDR3/LPDDR3"
    return "Unknown"

# Apply parsers
df["RAM_GB"] = df["Specifications"].apply(parse_ram)
df["Storage_GB"] = df["Specifications"].apply(parse_storage)
df["CPU_Brand"] = df["Specifications"].apply(parse_cpu_brand)
df["Processor_Name"] = df["Specifications"].apply(parse_processor)
df["Display_Size_Inch"] = df["Specifications"].apply(parse_display)
df["Touchscreen"] = df["Specifications"].apply(parse_touch)
df["Operating_System"] = df["Specifications"].apply(parse_os)
df["RAM_Type"] = df["Specifications"].apply(parse_ram_type)



In [36]:
df.head()

,Brand_Name,Price,RAM_GB,Storage_GB,CPU_Brand,Display_Size_Inch,Touchscreen,Operating_System,Rating,Ratings_Count,Reviews_Count,Processor_Name,RAM_Type,Price_Tier,RAM_Category,Storage_Category,Product_Title,Specifications
0,ASUS,62990.0,512.0,528.0,Intel,16.0,No,Windows,4.3,1318,74,Intel Core i5 Processor,DDR4/LPDDR4,None,None,None,ASUS Vivobook 16X (2025) for Creator with Offi...,Intel Core i5 Processor (13th Gen) | 16 GB DDR...
1,ASUS,66990.0,512.0,528.0,AMD,15.6,No,Windows,4.4,3096,169,AMD Ryzen 7 Hexa Core Processor,DDR5/LPDDR5,None,None,None,ASUS TUF Gaming A15 (2025) AMD Ryzen 7 Hexa Co...,AMD Ryzen 7 Hexa Core Processor | 16 GB DDR5 R...
2,ASUS,32190.0,512.0,520.0,AMD,15.6,No,Windows,4.3,1747,116,AMD Ryzen 3 Quad Core Processor,DDR5/LPDDR5,None,None,None,ASUS Vivobook Go 15 AMD Ryzen 3 Quad Core 7320...,AMD Ryzen 3 Quad Core Processor | 8 GB LPDDR5 ...
3,Lenovo,108990.0,32.0,1056.0,Intel,14.0,No,Windows,4.2,6,0,Intel Core Ultra 9 Processor,DDR5/LPDDR5,None,None,None,Lenovo IdeaPad Pro 5 Intel Core Ultra 9 285H -...,Intel Core Ultra 9 Processor | 32 GB LPDDR5X R...
4,Acer,23590.0,512.0,520.0,Intel,11.6,No,Windows,3.8,8033,700,Intel Celeron Dual Core Processor,DDR4/LPDDR4,None,None,None,Acer Aspire 3 Intel Celeron Dual Core - (8 GB/...,Intel Celeron Dual Core Processor | 8 GB DDR4 ...


In [37]:
# Desired column order
ordered_cols = [
    "Brand_Name",        # brand at the very beginning
    "Price",
    "RAM_GB",
    "Storage_GB",
    "CPU_Brand",
    "Display_Size_Inch",
    "Touchscreen",
    "Operating_System",
    "Rating",
    "Ratings_Count",
    "Reviews_Count",
    "Processor_Name",
    "RAM_Type",
    "Price_Tier",
    "RAM_Category",
    "Storage_Category",
    # reference variables at the end
    "Product_Title",
    "Specifications"
]

# Reorder DataFrame
df = df[ordered_cols]


In [38]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 984 entries, 0 to 983
Data columns (total 18 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Brand_Name         984 non-null    object 
 1   Price              984 non-null    float64
 2   RAM_GB             984 non-null    float64
 3   Storage_GB         984 non-null    float64
 4   CPU_Brand          984 non-null    object 
 5   Display_Size_Inch  982 non-null    float64
 6   Touchscreen        984 non-null    object 
 7   Operating_System   984 non-null    object 
 8   Rating             981 non-null    object 
 9   Ratings_Count      984 non-null    int64  
 10  Reviews_Count      984 non-null    int64  
 11  Processor_Name     984 non-null    object 
 12  RAM_Type           984 non-null    object 
 13  Price_Tier         0 non-null      object 
 14  RAM_Category       0 non-null      object 
 15  Storage_Category   0 non-null      object 
 16  Product_Title      984 non

In [39]:
df.head()

,Brand_Name,Price,RAM_GB,Storage_GB,CPU_Brand,Display_Size_Inch,Touchscreen,Operating_System,Rating,Ratings_Count,Reviews_Count,Processor_Name,RAM_Type,Price_Tier,RAM_Category,Storage_Category,Product_Title,Specifications
0,ASUS,62990.0,512.0,528.0,Intel,16.0,No,Windows,4.3,1318,74,Intel Core i5 Processor,DDR4/LPDDR4,None,None,None,ASUS Vivobook 16X (2025) for Creator with Offi...,Intel Core i5 Processor (13th Gen) | 16 GB DDR...
1,ASUS,66990.0,512.0,528.0,AMD,15.6,No,Windows,4.4,3096,169,AMD Ryzen 7 Hexa Core Processor,DDR5/LPDDR5,None,None,None,ASUS TUF Gaming A15 (2025) AMD Ryzen 7 Hexa Co...,AMD Ryzen 7 Hexa Core Processor | 16 GB DDR5 R...
2,ASUS,32190.0,512.0,520.0,AMD,15.6,No,Windows,4.3,1747,116,AMD Ryzen 3 Quad Core Processor,DDR5/LPDDR5,None,None,None,ASUS Vivobook Go 15 AMD Ryzen 3 Quad Core 7320...,AMD Ryzen 3 Quad Core Processor | 8 GB LPDDR5 ...
3,Lenovo,108990.0,32.0,1056.0,Intel,14.0,No,Windows,4.2,6,0,Intel Core Ultra 9 Processor,DDR5/LPDDR5,None,None,None,Lenovo IdeaPad Pro 5 Intel Core Ultra 9 285H -...,Intel Core Ultra 9 Processor | 32 GB LPDDR5X R...
4,Acer,23590.0,512.0,520.0,Intel,11.6,No,Windows,3.8,8033,700,Intel Celeron Dual Core Processor,DDR4/LPDDR4,None,None,None,Acer Aspire 3 Intel Celeron Dual Core - (8 GB/...,Intel Celeron Dual Core Processor | 8 GB DDR4 ...


In [30]:
# --- Final Deduplication Strategy in One Cell ---

# Work on a copy for safety
df_clean = df.copy()

# 1. Drop exact duplicates across all columns
df_clean = df_clean.drop_duplicates(keep='first').reset_index(drop=True)

# 2. Create a helper key for Product_Title + Specifications
df_clean["title_specs_key"] = (
    df_clean["Product_Title"].str.lower().str.strip() + " || " +
    df_clean["Specifications"].str.lower().str.strip()
)

# 3. Drop duplicates only if ALL values match (title+specs+price+ratings+reviews)
#    → preserves seller variation
df_clean = df_clean.drop_duplicates(
    subset=["title_specs_key","Price","Rating","Ratings_Count","Reviews_Count"],
    keep='first'
).reset_index(drop=True)

# 4. Remove helper column
df_clean = df_clean.drop(columns=["title_specs_key"])

# 5. Audit after deduplication
print("Final row count:", len(df_clean))
print("Remaining duplicates by title+specs:",
      df_clean.duplicated(subset=["Product_Title","Specifications"]).sum())


Final row count: 569
Remaining duplicates by title+specs: 14


In [40]:
df_clean.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 569 entries, 0 to 568
Data columns (total 18 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Brand_Name         569 non-null    object 
 1   Price              569 non-null    float64
 2   RAM_GB             569 non-null    float64
 3   Storage_GB         569 non-null    float64
 4   CPU_Brand          569 non-null    object 
 5   Display_Size_Inch  567 non-null    float64
 6   Touchscreen        569 non-null    object 
 7   Operating_System   569 non-null    object 
 8   Rating             566 non-null    object 
 9   Ratings_Count      569 non-null    int64  
 10  Reviews_Count      569 non-null    int64  
 11  Processor_Name     569 non-null    object 
 12  RAM_Type           569 non-null    object 
 13  Price_Tier         0 non-null      object 
 14  RAM_Category       0 non-null      object 
 15  Storage_Category   0 non-null      object 
 16  Product_Title      569 non

In [41]:
df_clean.head()

,Brand_Name,Price,RAM_GB,Storage_GB,CPU_Brand,Display_Size_Inch,Touchscreen,Operating_System,Rating,Ratings_Count,Reviews_Count,Processor_Name,RAM_Type,Price_Tier,RAM_Category,Storage_Category,Product_Title,Specifications
0,ASUS,62990.0,512.0,528.0,Intel,16.0,No,Windows,4.3,1318,74,Intel Core i5 Processor,DDR4/LPDDR4,None,None,None,ASUS Vivobook 16X (2025) for Creator with Offi...,Intel Core i5 Processor (13th Gen) | 16 GB DDR...
1,ASUS,66990.0,512.0,528.0,AMD,15.6,No,Windows,4.4,3096,169,AMD Ryzen 7 Hexa Core Processor,DDR5/LPDDR5,None,None,None,ASUS TUF Gaming A15 (2025) AMD Ryzen 7 Hexa Co...,AMD Ryzen 7 Hexa Core Processor | 16 GB DDR5 R...
2,ASUS,32190.0,512.0,520.0,AMD,15.6,No,Windows,4.3,1747,116,AMD Ryzen 3 Quad Core Processor,DDR5/LPDDR5,None,None,None,ASUS Vivobook Go 15 AMD Ryzen 3 Quad Core 7320...,AMD Ryzen 3 Quad Core Processor | 8 GB LPDDR5 ...
3,Lenovo,108990.0,32.0,1056.0,Intel,14.0,No,Windows,4.2,6,0,Intel Core Ultra 9 Processor,DDR5/LPDDR5,None,None,None,Lenovo IdeaPad Pro 5 Intel Core Ultra 9 285H -...,Intel Core Ultra 9 Processor | 32 GB LPDDR5X R...
4,Acer,23590.0,512.0,520.0,Intel,11.6,No,Windows,3.8,8033,700,Intel Celeron Dual Core Processor,DDR4/LPDDR4,None,None,None,Acer Aspire 3 Intel Celeron Dual Core - (8 GB/...,Intel Celeron Dual Core Processor | 8 GB DDR4 ...


In [43]:
# --- Step: Missing Value Handling ---

import numpy as np
import pandas as pd

# Ensure numeric columns are numeric
numeric_cols = ["Price","RAM_GB","Storage_GB","Display_Size_Inch","Rating","Ratings_Count","Reviews_Count"]
for c in numeric_cols:
    if c in df_clean.columns:
        df_clean[c] = pd.to_numeric(df_clean[c], errors="coerce")

# Chromebooks storage rule: if OS is Chrome OS and Storage_GB is missing, set to 0
mask_chrome_unknown = (df_clean["Operating_System"].str.contains("Chrome", case=False, na=False)) & (df_clean["Storage_GB"].isna())
df_clean.loc[mask_chrome_unknown, "Storage_GB"] = 0

# Median imputation for remaining numeric nulls
for c in numeric_cols:
    if c in df_clean.columns:
        med = df_clean[c].median(skipna=True)
        df_clean[c] = df_clean[c].fillna(med)

# Categorical nulls → "Unknown"
categorical_cols = ["Brand_Name","CPU_Brand","Touchscreen","Operating_System","Processor_Name","RAM_Type"]
for c in categorical_cols:
    if c in df_clean.columns:
        df_clean[c] = df_clean[c].fillna("Unknown")

# Audit after missing value handling
print("Row count:", len(df_clean))
print("Null counts:\n", df_clean.isna().sum())


Row count: 569
Null counts:
 Brand_Name             0
Price                  0
RAM_GB                 0
Storage_GB             0
CPU_Brand              0
Display_Size_Inch      0
Touchscreen            0
Operating_System       0
Rating                 0
Ratings_Count          0
Reviews_Count          0
Processor_Name         0
RAM_Type               0
Price_Tier           569
RAM_Category         569
Storage_Category     569
Product_Title          0
Specifications         0
dtype: int64


In [44]:
df_clean.head()

,Brand_Name,Price,RAM_GB,Storage_GB,CPU_Brand,Display_Size_Inch,Touchscreen,Operating_System,Rating,Ratings_Count,Reviews_Count,Processor_Name,RAM_Type,Price_Tier,RAM_Category,Storage_Category,Product_Title,Specifications
0,ASUS,62990.0,512.0,528.0,Intel,16.0,No,Windows,4.3,1318,74,Intel Core i5 Processor,DDR4/LPDDR4,None,None,None,ASUS Vivobook 16X (2025) for Creator with Offi...,Intel Core i5 Processor (13th Gen) | 16 GB DDR...
1,ASUS,66990.0,512.0,528.0,AMD,15.6,No,Windows,4.4,3096,169,AMD Ryzen 7 Hexa Core Processor,DDR5/LPDDR5,None,None,None,ASUS TUF Gaming A15 (2025) AMD Ryzen 7 Hexa Co...,AMD Ryzen 7 Hexa Core Processor | 16 GB DDR5 R...
2,ASUS,32190.0,512.0,520.0,AMD,15.6,No,Windows,4.3,1747,116,AMD Ryzen 3 Quad Core Processor,DDR5/LPDDR5,None,None,None,ASUS Vivobook Go 15 AMD Ryzen 3 Quad Core 7320...,AMD Ryzen 3 Quad Core Processor | 8 GB LPDDR5 ...
3,Lenovo,108990.0,32.0,1056.0,Intel,14.0,No,Windows,4.2,6,0,Intel Core Ultra 9 Processor,DDR5/LPDDR5,None,None,None,Lenovo IdeaPad Pro 5 Intel Core Ultra 9 285H -...,Intel Core Ultra 9 Processor | 32 GB LPDDR5X R...
4,Acer,23590.0,512.0,520.0,Intel,11.6,No,Windows,3.8,8033,700,Intel Celeron Dual Core Processor,DDR4/LPDDR4,None,None,None,Acer Aspire 3 Intel Celeron Dual Core - (8 GB/...,Intel Celeron Dual Core Processor | 8 GB DDR4 ...


In [45]:
# Check rows where categorical fields were filled with "Unknown"
unknown_mask = (
    (df_clean["Brand_Name"] == "Unknown") |
    (df_clean["CPU_Brand"] == "Unknown") |
    (df_clean["Touchscreen"] == "Unknown") |
    (df_clean["Operating_System"] == "Unknown") |
    (df_clean["Processor_Name"] == "Unknown") |
    (df_clean["RAM_Type"] == "Unknown")
)

df_unknowns = df_clean[unknown_mask]

print("Rows with 'Unknown' values:", len(df_unknowns))


Rows with 'Unknown' values: 30


In [48]:
df_unknowns

,Brand_Name,Price,RAM_GB,Storage_GB,CPU_Brand,Display_Size_Inch,Touchscreen,Operating_System,Rating,Ratings_Count,Reviews_Count,Processor_Name,RAM_Type,Price_Tier,RAM_Category,Storage_Category,Product_Title,Specifications
30,Lenovo,11990.0,4.0,4.0,Unknown,11.6,No,Chrome OS,4.0,2339,170,Unknown,DDR4/LPDDR4,None,None,None,Lenovo 100e Chromebook Gen 4 MediaTek Kompanio...,MediaTek Kompanio 520 Processor | 4 GB LPDDR4X...
65,Primebook,22490.0,8.0,8.0,Unknown,15.6,No,Unknown,4.3,432,137,Unknown,DDR4/LPDDR4,None,None,None,Primebook 2 Max (2025) in-Built AI MediaTek He...,MediaTek Helio G99 (MT8781) Processor | 8 GB L...
77,Lenovo,14999.0,4.0,4.0,Unknown,14.0,No,Chrome OS,3.9,3523,313,Unknown,DDR4/LPDDR4,None,None,None,Lenovo Chromebook MediaTek Kompanio 520 - (4 G...,MediaTek Kompanio 520 Processor | 4 GB LPDDR4X...
96,Apple,91990.0,256.0,272.0,Apple,13.6,No,macOS,4.7,2268,155,Apple M4,Unknown,None,None,None,Apple MacBook Air M4 - (16 GB/256 GB SSD/macOS...,Apple M4 Processor | 16 GB Unified Memory RAM ...
104,Apple,91990.0,256.0,272.0,Apple,13.6,No,macOS,4.7,2268,155,Apple M4,Unknown,None,None,None,Apple MacBook Air M4 - (16 GB/256 GB SSD/macOS...,Apple M4 Processor | 16 GB Unified Memory RAM ...
139,Primebook,18990.0,8.0,8.0,Unknown,14.1,No,Unknown,4.3,432,137,Unknown,DDR4/LPDDR4,None,None,None,Primebook 2 Pro (2025) in-Built AI MediaTek He...,MediaTek Helio G99 (MT8781) Processor | 8 GB L...
148,Apple,114990.0,512.0,528.0,Apple,13.6,No,macOS,4.7,2268,155,Apple M4,Unknown,None,None,None,Apple MacBook Air M4 - (16 GB/512 GB SSD/macOS...,Apple M4 Processor | 16 GB Unified Memory RAM ...
182,Apple,372990.0,48.0,1072.0,Apple,16.0,No,macOS,4.0,4,1,Apple M3,Unknown,None,None,None,Apple MacBook Pro M3 Max - (48 GB/1 TB SSD/mac...,Apple M3 Max Processor | 48 GB Unified Memory ...
187,Apple,91990.0,256.0,272.0,Apple,13.6,No,macOS,4.7,2268,155,Apple M4,Unknown,None,None,None,Apple MacBook Air M4 - (16 GB/256 GB SSD/macOS...,Apple M4 Processor | 16 GB Unified Memory RAM ...
190,Lenovo,24899.0,8.0,8.0,Unknown,14.0,No,Chrome OS,3.9,3523,313,Unknown,DDR4/LPDDR4,None,None,None,Lenovo Chromebook MediaTek Kompanio 520 - (8 G...,MediaTek Kompanio 520 Processor | 8 GB LPDDR4X...


In [49]:
# Count how many "Unknown" values are present in each categorical column
categorical_cols = ["Brand_Name","CPU_Brand","Touchscreen","Operating_System","Processor_Name","RAM_Type"]

unknown_summary = {}
for col in categorical_cols:
    unknown_summary[col] = (df_clean[col] == "Unknown").sum()

print("Summary of 'Unknown' values per column:")
print(pd.Series(unknown_summary))

Summary of 'Unknown' values per column:
Brand_Name           0
CPU_Brand            7
Touchscreen          0
Operating_System     2
Processor_Name       7
RAM_Type            23
dtype: int64


In [52]:
# Filter rows where any categorical column has "Unknown"
df_unknowns = df_clean[(df_clean[categorical_cols] == "Unknown").any(axis=1)]

print("Rows with Unknown values:", len(df_unknowns))


Rows with Unknown values: 30


In [53]:
# --- Documentation: Missing Value Strategy ---
print("""
Missing values were handled as follows:
- Numerical columns: imputed with median (except Chromebooks storage → 0).
- Categorical columns: replaced with 'Unknown' for transparency.
- Preserved rows instead of dropping them, ensuring dataset size remains strong (569 rows).
""")

# Quick summary of Unknown counts
print("Unknown counts per column:")
print(pd.Series({
    "CPU_Brand": (df_clean["CPU_Brand"] == "Unknown").sum(),
    "Operating_System": (df_clean["Operating_System"] == "Unknown").sum(),
    "Processor_Name": (df_clean["Processor_Name"] == "Unknown").sum(),
    "RAM_Type": (df_clean["RAM_Type"] == "Unknown").sum()
}))


Missing values were handled as follows:
- Numerical columns: imputed with median (except Chromebooks storage → 0).
- Categorical columns: replaced with 'Unknown' for transparency.
- Preserved rows instead of dropping them, ensuring dataset size remains strong (569 rows).

Unknown counts per column:
CPU_Brand            7
Operating_System     2
Processor_Name       7
RAM_Type            23
dtype: int64


- Scraped 984 raw rows and reshaped them into a clean schema with 18 target columns.
- Converted messy strings (₹, commas in counts) into proper numeric types.
- Extracted Brand_Name from product titles.
- Parsed specifications with regex to fill RAM, Storage, CPU, OS, Display size, Touchscreen, and RAM type.
- Applied a deduplication strategy: dropped 415 exact duplicates, preserved seller variation, final dataset = 569 rows.
- Handled missing values systematically, leaving placeholders (Unknown) for categorical gaps.


#### Why so many "Unknown" values
- Some laptops (e.g., Primebook, Chromebooks, Apple M‑series) have specs written in non‑standard formats.
- Regex could not always capture CPU brand or RAM type (e.g., “Unified Memory” in Apple devices).
- Certain OS fields were incomplete or missing in the raw scrape.
- Instead of discarding these rows, we marked them as "Unknown" to keep the dataset size strong and analysis reproducible.


In [56]:
# --- Impute Chrome OS and default storage ---

# Copy into a new frame for safety
df_os_storage = df_clean.copy()

# 1. Impute missing Operating_System as "Chrome OS"
df_os_storage["Operating_System"] = df_os_storage["Operating_System"].replace("Unknown", "Chrome OS")

# 2. For rows where Operating_System is Chrome OS and Storage_GB is very small or missing, set default
# Let's assume default = 64 GB (common Chromebook eMMC size)
mask_chrome_storage = (df_os_storage["Operating_System"] == "Chrome OS") & (df_os_storage["Storage_GB"] <= 8)
df_os_storage.loc[mask_chrome_storage, "Storage_GB"] = 64

# 3. Audit how many values were imputed
os_filled = (df_clean["Operating_System"] == "Unknown").sum()
storage_filled = mask_chrome_storage.sum()

print("Operating_System imputed to Chrome OS:", os_filled)
print("Storage_GB imputed to 64 GB for Chrome OS rows:", storage_filled)

# Preview the affected rows
print(df_os_storage[mask_chrome_storage].head())

Operating_System imputed to Chrome OS: 2
Storage_GB imputed to 64 GB for Chrome OS rows: 23
   Brand_Name    Price  RAM_GB  Storage_GB CPU_Brand  Display_Size_Inch  \
28       ASUS  17990.0     4.0        64.0     Intel               15.6   
30     Lenovo  11990.0     4.0        64.0   Unknown               11.6   
42       Acer  23990.0     8.0        64.0     Intel               15.6   
43       ASUS  14990.0     4.0        64.0     Intel               15.6   
65  Primebook  22490.0     8.0        64.0   Unknown               15.6   

   Touchscreen Operating_System  Rating  Ratings_Count  Reviews_Count  \
28          No        Chrome OS     4.1             63              5   
30          No        Chrome OS     4.0           2339            170   
42          No        Chrome OS     3.8           2681            187   
43          No        Chrome OS     3.8            132              6   
65          No        Chrome OS     4.3            432            137   

                  


#  Null Value Handling – Documentation

### Approach
- **Numerical columns**  
  - Converted to proper numeric types.  
  - Remaining nulls imputed with **median values** for stability and to avoid skewing distributions.  

- **Chromebooks – Special Rule **  
  - If `Operating_System` was missing, imputed as **"Chrome OS"** (since these devices are clearly Chromebooks/Primebooks).  
  - If `Storage_GB` was missing or unrealistically small (≤8 GB), imputed as **64 GB**, reflecting typical eMMC defaults.  
  - This ensures consistency in categories and avoids misleading `"Unknown"` or `0 GB` values.  

- **Categorical columns**  
  - Missing values replaced with **"Unknown"** for transparency.  
  - This makes gaps visible during analysis instead of silently dropping rows.  

### Outcome
- **Row count preserved:** 569 rows remain after cleaning.  
- **No nulls left:** all columns are complete.  
- **Transparency maintained:** `"Unknown"` placeholders show where the scrape lacked information.  
- **Assumptions documented:** mentors/recruiters can trust the cleaning logic because imputations are explicit and justified.  


## 3. Validation Strategy
- After each step: print shape, null counts, duplicates, dtypes, and sample rows.
- Ensures reproducibility and transparency.
- Audit trail shows mentor/recruiter that every choice is defensible.

---

## 4. EDA Plan
### Univariate
- Histograms for Price, RAM, Storage, Ratings.
- Countplots for Brand, OS, CPU, Price_Tier.

### Bivariate
- Scatterplots: Price vs RAM/Storage/Rating.
- Boxplots: Price by RAM_Category, Storage_Category.

### Multivariate
- Heatmap: Median Price by Brand × CPU × RAM_Category.
- Ratings vs OS and Price_Tier.

---

## 5. Presentation Notes
- **Explain logic step‑by‑step**: “We first renamed columns, then parsed specs with regex, then engineered features.”
- **Highlight reproducibility**: “Every step has validation prints—mentors can trust the pipeline.”
- **Business insight angle**: “EDA shows which brands dominate mid‑range, which configs get higher ratings.”
- **Interview prep**: Emphasize handling messy real‑world data, regex extraction, feature engineering, and clear documentation.

---
Future Improvements
- Expand regex to capture edge cases (dual storage, LPDDR variants).
- Automate thresholds for tiers based on quantiles instead of fixed values.
- Add visualization filtering (top brands only, avoid clutter).
- Export clean dataset to CSV for reuse in SQL/Power BI.


In [57]:
# --- Step: Feature Engineering (with Chrome OS imputation already applied) ---

# 1. Price Tier
def price_to_tier(price):
    if pd.isna(price): return "Unknown"
    if price < 45000: return "Budget"
    if 45000 <= price < 80000: return "Mid-range"
    return "Premium"

df_os_storage["Price_Tier"] = df_os_storage["Price"].apply(price_to_tier)

# 2. RAM Category
def ram_to_category(ram_gb):
    if pd.isna(ram_gb): return "Unknown"
    if ram_gb < 8: return "Low"
    if 8 <= ram_gb <= 12: return "Mid"
    return "High"

df_os_storage["RAM_Category"] = df_os_storage["RAM_GB"].apply(ram_to_category)

# 3. Storage Category
def storage_to_category(storage_gb):
    if pd.isna(storage_gb): return "Unknown"
    if storage_gb < 512: return "Low"
    if storage_gb == 512: return "Mid"
    if storage_gb >= 1024: return "High"
    return "High"

df_os_storage["Storage_Category"] = df_os_storage["Storage_GB"].apply(storage_to_category)

# 4. Audit after feature engineering
print("Row count:", len(df_os_storage))


Row count: 569


In [58]:
df_os_storage.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 569 entries, 0 to 568
Data columns (total 18 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Brand_Name         569 non-null    object 
 1   Price              569 non-null    float64
 2   RAM_GB             569 non-null    float64
 3   Storage_GB         569 non-null    float64
 4   CPU_Brand          569 non-null    object 
 5   Display_Size_Inch  569 non-null    float64
 6   Touchscreen        569 non-null    object 
 7   Operating_System   569 non-null    object 
 8   Rating             569 non-null    float64
 9   Ratings_Count      569 non-null    int64  
 10  Reviews_Count      569 non-null    int64  
 11  Processor_Name     569 non-null    object 
 12  RAM_Type           569 non-null    object 
 13  Price_Tier         569 non-null    object 
 14  RAM_Category       569 non-null    object 
 15  Storage_Category   569 non-null    object 
 16  Product_Title      569 non

In [59]:
df_os_storage.head()

,Brand_Name,Price,RAM_GB,Storage_GB,CPU_Brand,Display_Size_Inch,Touchscreen,Operating_System,Rating,Ratings_Count,Reviews_Count,Processor_Name,RAM_Type,Price_Tier,RAM_Category,Storage_Category,Product_Title,Specifications
0,ASUS,62990.0,512.0,528.0,Intel,16.0,No,Windows,4.3,1318,74,Intel Core i5 Processor,DDR4/LPDDR4,Mid-range,High,High,ASUS Vivobook 16X (2025) for Creator with Offi...,Intel Core i5 Processor (13th Gen) | 16 GB DDR...
1,ASUS,66990.0,512.0,528.0,AMD,15.6,No,Windows,4.4,3096,169,AMD Ryzen 7 Hexa Core Processor,DDR5/LPDDR5,Mid-range,High,High,ASUS TUF Gaming A15 (2025) AMD Ryzen 7 Hexa Co...,AMD Ryzen 7 Hexa Core Processor | 16 GB DDR5 R...
2,ASUS,32190.0,512.0,520.0,AMD,15.6,No,Windows,4.3,1747,116,AMD Ryzen 3 Quad Core Processor,DDR5/LPDDR5,Budget,High,High,ASUS Vivobook Go 15 AMD Ryzen 3 Quad Core 7320...,AMD Ryzen 3 Quad Core Processor | 8 GB LPDDR5 ...
3,Lenovo,108990.0,32.0,1056.0,Intel,14.0,No,Windows,4.2,6,0,Intel Core Ultra 9 Processor,DDR5/LPDDR5,Premium,High,High,Lenovo IdeaPad Pro 5 Intel Core Ultra 9 285H -...,Intel Core Ultra 9 Processor | 32 GB LPDDR5X R...
4,Acer,23590.0,512.0,520.0,Intel,11.6,No,Windows,3.8,8033,700,Intel Celeron Dual Core Processor,DDR4/LPDDR4,Budget,High,High,Acer Aspire 3 Intel Celeron Dual Core - (8 GB/...,Intel Celeron Dual Core Processor | 8 GB DDR4 ...


In [62]:
df_os_storage.isnull().any().isnull().any()

Brand_Name           False
Price                False
RAM_GB               False
Storage_GB           False
CPU_Brand            False
Display_Size_Inch    False
Touchscreen          False
Operating_System     False
Rating               False
Ratings_Count        False
Reviews_Count        False
Processor_Name       False
RAM_Type             False
Price_Tier           False
RAM_Category         False
Storage_Category     False
Product_Title        False
Specifications       False
dtype: bool

print("\nFinal validation snapshot:")
print("Shape:", df_final.shape)
print("\nDtypes:\n", df_final.dtypes)
print("\nNull counts:\n", df_final.isnull().sum())
print("\nRemaining exact duplicates:", df_final.duplicated().sum())
print("\nSample rows:\n", df_final.head(5))

# Save outputs for EDA and reproducibility
df_final.to_csv("flipkart_laptops_clean.csv", index=False)
print("\nSaved: flipkart_laptops_clean.csv")

In [63]:
# --- Final Professional Audit for df_os_storage ---

print("\nFinal validation snapshot:")
print("Shape:", df_os_storage.shape)

print("\nDtypes:\n", df_os_storage.dtypes)

print("\nNull counts:\n", df_os_storage.isnull().sum())

print("\nRemaining exact duplicates:", df_os_storage.duplicated().sum())

print("\nRemaining title+specs duplicates:",
      df_os_storage.duplicated(subset=["Product_Title","Specifications"]).sum())

print("\nUnique values check (categorical sanity):")
for col in ["Brand_Name","CPU_Brand","Operating_System","RAM_Type","Price_Tier","RAM_Category","Storage_Category"]:
    print(f"{col}: {df_os_storage[col].nunique()} unique values")

print("\nSample rows:\n", df_os_storage.head(5))



Final validation snapshot:
Shape: (569, 18)

Dtypes:
 Brand_Name            object
Price                float64
RAM_GB               float64
Storage_GB           float64
CPU_Brand             object
Display_Size_Inch    float64
Touchscreen           object
Operating_System      object
Rating               float64
Ratings_Count          int64
Reviews_Count          int64
Processor_Name        object
RAM_Type              object
Price_Tier            object
RAM_Category          object
Storage_Category      object
Product_Title         object
Specifications        object
dtype: object

Null counts:
 Brand_Name           0
Price                0
RAM_GB               0
Storage_GB           0
CPU_Brand            0
Display_Size_Inch    0
Touchscreen          0
Operating_System     0
Rating               0
Ratings_Count        0
Reviews_Count        0
Processor_Name       0
RAM_Type             0
Price_Tier           0
RAM_Category         0
Storage_Category     0
Product_Title        0
Spe

In [66]:

# Unique values per column
for col in df_os_storage.columns:
    print(f"{col}: {df_os_storage[col].nunique()} unique values")

Brand_Name: 21 unique values
Price: 313 unique values
RAM_GB: 12 unique values
Storage_GB: 27 unique values
CPU_Brand: 5 unique values
Display_Size_Inch: 17 unique values
Touchscreen: 2 unique values
Operating_System: 3 unique values
Rating: 24 unique values
Ratings_Count: 304 unique values
Reviews_Count: 142 unique values
Processor_Name: 37 unique values
RAM_Type: 4 unique values
Price_Tier: 3 unique values
RAM_Category: 3 unique values
Storage_Category: 2 unique values
Product_Title: 542 unique values
Specifications: 505 unique values


In [67]:
# --- Unique values and their counts for each column ---

for col in df_os_storage.columns:
    print(f"\nColumn: {col}")
    print(df_os_storage[col].value_counts())


Column: Brand_Name
Brand_Name
ASUS         128
HP           115
Lenovo        88
Acer          74
DELL          53
MSI           50
Apple         24
Samsung       10
Infinix        9
Colorful       4
GIGABYTE       2
MOTOROLA       2
Primebook      2
Prittec        1
walker         1
WINGS          1
Thomson        1
ZEBRONICS      1
Avita          1
MICROSOFT      1
realme         1
Name: count, dtype: int64

Column: Price
Price
42990.0    11
56990.0    11
37990.0    11
59990.0    11
72990.0     9
           ..
45497.0     1
70999.0     1
41445.0     1
37499.0     1
41144.0     1
Name: count, Length: 313, dtype: int64

Column: RAM_GB
RAM_GB
512.0    399
16.0      60
256.0     37
32.0      25
8.0       19
4.0       13
24.0       8
36.0       3
128.0      2
48.0       1
18.0       1
64.0       1
Name: count, dtype: int64

Column: Storage_GB
Storage_GB
528.0     238
520.0     139
1040.0     59
64.0       23
1056.0     21
264.0      19
536.0       9
1032.0      8
1048.0      8
272.0     

#  Column Descriptions & Unique Values

| Column             | Unique Values |  Frequent Values                                                                 | Rare                                                                 |
|--------------------|---------------|------------------------------------------------------------------------------------------|-------------------------------------------------------------------------------|
| **Brand_Name**     | 21            | ASUS (128), HP (115), Lenovo (88), Acer (74), DELL (53), MSI (50)                        | Rare brands: MICROSOFT, realme, Avita, Thomson, etc.                          |
| **Price**          | 313           | Range: ~₹11,990 to ~₹416,000                                                             | Wide spread across tiers                                                      |
| **RAM_GB**         | 12            | 512 GB (399), 16 GB (60), 256 GB (37)                                                    | Rare: 18 GB, 48 GB, 64 GB                                                     |
| **Storage_GB**     | 27            | 528 GB (238), 520 GB (139), 1040 GB (59)                                                 | Defaults: 64 GB (23 Chromebooks imputed). Rare: 4160 GB, 2064 GB, 1296 GB. Formula used: **1 TB = 1024 GB** |
| **CPU_Brand**      | 5             | Intel (402), AMD (123), Apple (23)                                                       | Qualcomm (14), Unknown (7)                                                    |
| **Display_Size_Inch** | 17         | 15.6" (305), 14.0" (154), 16.0" (57)                                                     | Rare: 12.4", 14.9", 18.0"                                                     |
| **Touchscreen**    | 2             | No (542)                                                                                 | Yes (27)                                                                      |
| **Operating_System** | 3           | Windows (522)                                                                            | macOS (24), Chrome OS (23)                                                    |
| **Rating**         | 24            | Spread across 3.5–4.5 range                                                              | Few laptops with extreme ratings                                              |
| **Ratings_Count**  | 304           | Highly varied, from single digits to thousands                                           | —                                                                             |
| **Reviews_Count**  | 142           | Highly varied, from 0 to hundreds                                                        | —                                                                             |
| **Processor_Name** | 37            | Intel Core i5 (142), Intel Core i3 (86), Intel Core i7 (57), AMD Ryzen 5 Hexa Core (28)  | Apple M4 (13), Apple M3 (7), Apple M5 (1), Snapdragon X Elite (3), AMD Ryzen AI processors |
| **RAM_Type**       | 4             | DDR4/LPDDR4 (297), DDR5/LPDDR5 (248)                                                     | Unknown (23), DDR3/LPDDR3 (1)                                                 |
| **Price_Tier**     | 3             | Mid‑range (245), Budget (193), Premium (131)                                             | —                                                                             |
| **RAM_Category**   | 3             | High (537)                                                                               | Mid (19), Low (13)                                                            |
| **Storage_Category** | 2           | High (512)                                                                               | Low (57)                                                                      |
| **Product_Title**  | 542           | Some titles repeated (e.g., ASUS Zenbook A14 OLED → 4 times)                             | Majority occur once (long tail distribution)                                  |
| **Specifications** | 505           | Common configs repeated (e.g., AMD Ryzen 5 Quad Core | 8 GB RAM | 512 GB SSD → 6 times) | Majority occur once, reflecting diversity                                     |

In [78]:
# Count "Unknown" values across all columns
unknown_counts = (df_os_storage == "Unknown").sum()

print("Unknown values per column:\n", unknown_counts)

# Total "Unknown" values in the whole dataset
total_unknown = (df_os_storage == "Unknown").sum().sum()
print("\nTotal Unknown values in dataset:", total_unknown)

Unknown values per column:
 Brand_Name           0
Price                0
RAM_GB               0
Storage_GB           0
CPU_Brand            7
Display_Size_Inch    0
Touchscreen          0
Operating_System     0
Rating               0
Ratings_Count        0
Reviews_Count        0
Processor_Name       7
RAM_Type             0
Price_Tier           0
RAM_Category         0
Storage_Category     0
Product_Title        0
Specifications       0
dtype: int64

Total Unknown values in dataset: 14


In [76]:
# --- Show rows with "Unknown" values ---
mask_unknown = df_os_storage.isin(["Unknown"]).any(axis=1)
unknown_rows = df_os_storage[mask_unknown]

print("Rows with Unknown values:", len(unknown_rows))


Rows with Unknown values: 7


In [77]:
unknown_rows[["Brand_Name","CPU_Brand","Processor_Name","RAM_Type","Operating_System","Product_Title"]]

,Brand_Name,CPU_Brand,Processor_Name,RAM_Type,Operating_System,Product_Title
30,Lenovo,Unknown,Unknown,DDR4/LPDDR4,Chrome OS,Lenovo 100e Chromebook Gen 4 MediaTek Kompanio...
65,Primebook,Unknown,Unknown,DDR4/LPDDR4,Chrome OS,Primebook 2 Max (2025) in-Built AI MediaTek He...
77,Lenovo,Unknown,Unknown,DDR4/LPDDR4,Chrome OS,Lenovo Chromebook MediaTek Kompanio 520 - (4 G...
139,Primebook,Unknown,Unknown,DDR4/LPDDR4,Chrome OS,Primebook 2 Pro (2025) in-Built AI MediaTek He...
190,Lenovo,Unknown,Unknown,DDR4/LPDDR4,Chrome OS,Lenovo Chromebook MediaTek Kompanio 520 - (8 G...
289,Lenovo,Unknown,Unknown,DDR4/LPDDR4,Chrome OS,Lenovo Chromebook Duet 11M889 MediaTek MediaTe...
524,HP,Unknown,Unknown,DDR4/LPDDR4,Chrome OS,HP Touch Chromebook MediaTek MT8183 - (4 GB/32...


#### Update Apple MacBooks RAM_Type from "Unknown" to "Unified Memory"

In [75]:
# Update Apple MacBooks RAM_Type from "Unknown" to "Unified Memory"
df_os_storage.loc[
    (df_os_storage["Brand_Name"] == "Apple") & (df_os_storage["RAM_Type"] == "Unknown"),
    "RAM_Type"
] = "Unified Memory"

# Double-check the change
print(df_os_storage.loc[df_os_storage["Brand_Name"] == "Apple", ["Brand_Name","Processor_Name","RAM_Type"]].head(10))

    Brand_Name Processor_Name        RAM_Type
96       Apple       Apple M4  Unified Memory
104      Apple       Apple M4  Unified Memory
148      Apple       Apple M4  Unified Memory
182      Apple       Apple M3  Unified Memory
187      Apple       Apple M4  Unified Memory
201      Apple       Apple M3  Unified Memory
222      Apple       Apple M3  Unified Memory
224      Apple       Apple M5  Unified Memory
250      Apple       Apple M3  Unified Memory
252      Apple       Apple M3  Unified Memory


### Handling Unknowns in Chromebooks

- 7 rows (Lenovo, Primebook, HP) are **Chrome OS devices** with missing `CPU_Brand` and `Processor_Name`.  
- These were left as `"Unknown"` to preserve transparency, since Flipkart specs did not provide details.  
- `RAM_Type` values are valid (DDR4/LPDDR4).  
- Documented as expected edge cases rather than errors.  

###  Handling OS and Storage

- **Operating_System:**  
  - Missing OS values were imputed as `"Chrome OS"` for Chromebooks.  
  - Apple laptops correctly show `"macOS"`.  
  - Windows laptops were already consistent.  

- **Storage_GB:**  
  - Chromebooks with very small or missing storage (≤8 GB) were imputed to **64 GB**, reflecting typical eMMC defaults.  
  - All TB values were converted to GB using the formula **1 TB = 1024 GB**.  
  - Final storage categories: **High (≥1 TB)** and **Low (<512 GB)**, with 512 GB treated as the mid‑point.  


- Chromebooks (Lenovo, Primebook, HP) had missing CPU_Brand and Processor_Name. These were left as "Unknown" to preserve transparency, since Flipkart specs did not provide details.
- Apple MacBooks use Unified Memory instead of DDR types. RAM_Type values marked "Unknown" were updated to "Unified Memory" for accuracy.
- Total Unknowns after cleaning: 7 rows of chromebooks with missing cpu values


#  Flipkart Laptop Dataset – Final Documentation

### Dataset Size & Source
- Scraped ~984 raw rows from Flipkart product pages.
- After cleaning and deduplication, final dataset has **569 rows × 18 columns**.
- Exact duplicates removed: 415.
- Title+specs duplicates: 14 (kept intentionally to preserve seller variation).
- Saved as `flipkart_laptops_clean.csv` for reproducibility.

---

###  Column Descriptions & Unique Values
- **Brand_Name (21 unique)** → ASUS (128), HP (115), Lenovo (88), Acer (74), DELL (53), MSI (50), Apple (24), plus smaller brands like Samsung, Infinix, Avita, realme.  
- **Price (313 unique)** → ranges from ~₹11,990 to ~₹416,000.  
- **RAM_GB (12 unique)** → common: 512 GB (399 laptops), 16 GB (60), 256 GB (37). Rare: 18 GB, 48 GB, 64 GB.  
- **Storage_GB (27 unique)** → common: 528 GB (238), 520 GB (139), 1040 GB (59). Defaults: 64 GB (23 Chromebooks imputed). Formula used: **1 TB = 1024 GB**.  
- **CPU_Brand (5 unique)** → Intel (402), AMD (123), Apple (23), Qualcomm (14), Unknown (7).  
- **Display_Size_Inch (17 unique)** → common: 15.6" (305), 14.0" (154), 16.0" (57). Rare: 12.4", 14.9", 18.0".  
- **Touchscreen (2 unique)** → No (542), Yes (27).  
- **Operating_System (3 unique)** → Windows (522), macOS (24), Chrome OS (23).  
- **Processor_Name (37 unique)** → Intel Core i5 (142), Intel Core i3 (86), Intel Core i7 (57), AMD Ryzen 5 Hexa Core (28), Apple M4 (13), Apple M3 (7), plus rare entries like Snapdragon X Elite and AMD Ryzen AI processors.  
- **RAM_Type (4 unique)** → DDR4/LPDDR4 (297), DDR5/LPDDR5 (248), Unknown (23), DDR3/LPDDR3 (1).  
- **Price_Tier (3 unique)** → Budget (193), Mid‑range (245), Premium (131).  
- **RAM_Category (3 unique)** → High (537), Mid (19), Low (13).  
- **Storage_Category (2 unique)** → High (512), Low (57).  
- **Product_Title (542 unique)** → most titles occur once, but some repeat (e.g., ASUS Zenbook A14 OLED → 4 times).  
- **Specifications (505 unique)** → most specs occur once, but common configs repeat (e.g., AMD Ryzen 5 Quad Core | 8 GB RAM | 512 GB SSD → 6 times).

---

### 🔹 Null Values & Imputation
- Nulls found in CPU_Brand, Processor_Name, RAM_Type, Operating_System, Storage_GB.
- **Numerical columns:** imputed with median values for stability.
- **Chromebooks:**  
  - Missing OS → imputed as `"Chrome OS"`.  
  - Missing/very small storage (≤8 GB) → imputed as **64 GB** (default eMMC size).  
- **Categorical columns:** replaced with `"Unknown"` for transparency.
- **Outcome:** No nulls remain, dataset integrity preserved, assumptions documented.

---

### 🔹 Difficulties Faced
- Raw scrape had messy strings (`₹`, commas in counts).
- Apple laptops use “Unified Memory” instead of DDR types → harder to parse.
- Primebook entries had incomplete OS fields.
- Chromebooks often had missing storage or very small values.
- Seller variation caused duplicate titles/specs with different prices/ratings.

---

### Final Status
- Clean, deduplicated, imputed dataset.
- 569 rows × 18 columns.
- Rich diversity in brands, RAM, storage, processors, and OS.

In [80]:
df_os_storage.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 569 entries, 0 to 568
Data columns (total 18 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Brand_Name         569 non-null    object 
 1   Price              569 non-null    float64
 2   RAM_GB             569 non-null    float64
 3   Storage_GB         569 non-null    float64
 4   CPU_Brand          569 non-null    object 
 5   Display_Size_Inch  569 non-null    float64
 6   Touchscreen        569 non-null    object 
 7   Operating_System   569 non-null    object 
 8   Rating             569 non-null    float64
 9   Ratings_Count      569 non-null    int64  
 10  Reviews_Count      569 non-null    int64  
 11  Processor_Name     569 non-null    object 
 12  RAM_Type           569 non-null    object 
 13  Price_Tier         569 non-null    object 
 14  RAM_Category       569 non-null    object 
 15  Storage_Category   569 non-null    object 
 16  Product_Title      569 non

In [81]:

# Save outputs for reproducibility
df_os_storage.to_csv("flipkart_laptops_cleaned_data.csv", index=False)
print("\nSaved: flipkart_laptops_cleaned_data.csv")


Saved: flipkart_laptops_cleaned_data.csv
